# 06 — Train / Test Split Strategy (Leave-One-Cycle-Out)

**Goals**
- Implement leave-one-cycle-out splits separately for indoor and outdoor
- Verify no data leakage (whole cycle stays together)
- Save fold definitions for notebooks 07–08


In [ ]:
import pandas as pd
import json
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
df = pd.read_csv(PROCESSED_DIR / 'labeled_dataset.csv')
print('Cycles present:', df.cycle_id.unique().tolist())
print(df.groupby(['condition', 'cycle_id']).size())


In [ ]:
# Build LOLO folds
indoor_cycles  = sorted(df[df.condition == 'indoor']['cycle_id'].unique())
outdoor_cycles = sorted(df[df.condition == 'outdoor']['cycle_id'].unique())

folds = []
for test_c in indoor_cycles:
    folds.append({
        'fold_id': f'indoor_leave_{test_c}',
        'condition': 'indoor',
        'train_cycles': [c for c in indoor_cycles if c != test_c],
        'test_cycles': [test_c]
    })
for test_c in outdoor_cycles:
    folds.append({
        'fold_id': f'outdoor_leave_{test_c}',
        'condition': 'outdoor',
        'train_cycles': [c for c in outdoor_cycles if c != test_c],
        'test_cycles': [test_c]
    })

import pprint
pprint.pprint(folds)


In [ ]:
# Leakage check
for fold in folds:
    train_set = set(fold['train_cycles'])
    test_set  = set(fold['test_cycles'])
    assert train_set.isdisjoint(test_set)
    train_rows = df[df.cycle_id.isin(train_set)]
    test_rows  = df[df.cycle_id.isin(test_set)]
    assert set(train_rows.cycle_id) == train_set
    assert set(test_rows.cycle_id)  == test_set
print('Leakage check PASSED — cycles never split across train/test.')


In [ ]:
# Persist fold definitions
with open(PROCESSED_DIR / 'lolo_folds.json', 'w') as f:
    json.dump(folds, f, indent=2)
print('Saved → data/processed/lolo_folds.json')
print(f'Total folds: {len(folds)}')
